# Run all analysis notebooks

Runs every analysis notebook in this project in dependency order. Each notebook's code cells run top to bottom, like Jupyter. The source notebooks are **not** modified. Stops at the first failing cell and shows its traceback.

Notebook order (each step produces files the next one reads):
1. `1_eda.ipynb` → `dataset_v2.csv`
2. `2_cross_tab_analysis.ipynb` → `dataset_v3.csv`, `report_v2/`, `report_v2.2/`
3. `3_sig_analysis.ipynb` → `dataset_sig_st.csv`, `dataset_sig_phq9.csv`
4. `4_mlr_st.ipynb` → `report_v3/`
5. `5_mlr_phq9.ipynb` → `report_v3/`
6. `6_analysis_for_mind_well.ipynb` → `report_v4/`

In [4]:
import ast
import json
import os
import time
import traceback
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # figures go to PDFs; no GUI windows
from IPython.display import display

ROOT = Path.cwd()  # this notebook lives in the project root
os.chdir(ROOT)     # notebooks use relative paths such as "dataset_v2.csv"

NOTEBOOKS = [
    "1_eda.ipynb",
    "2_cross_tab_analysis.ipynb",
    "3_sig_analysis.ipynb",
    "4_mlr_st.ipynb",
    "5_mlr_phq9.ipynb",
    "6_analysis_for_mind_well.ipynb",
]

# Output folders used by the notebooks. Created up front because some notebooks
# write into them before their own os.makedirs call runs.
for folder in ["report_v2", "report_v2.2", "report_v3", "report_v4"]:
    os.makedirs(ROOT / folder, exist_ok=True)

In [5]:
import contextlib
import io
import warnings


def run_code_cell(source, namespace):
    """Execute one cell's code silently (no printed or displayed output)."""
    exec(compile(source, "<cell>", "exec"), namespace)


def run_notebook(path):
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)
    namespace = {"__name__": "__main__"}
    code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]
    for idx, cell in enumerate(code_cells, start=1):
        source = "".join(cell["source"])
        if not source.strip():
            continue
        buffer = io.StringIO()
        try:
            with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer), warnings.catch_warnings():
                warnings.simplefilter("ignore")
                run_code_cell(source, namespace)
        except Exception:
            print(f"[FAILED] {path.name}, code cell {idx} of {len(code_cells)}:")
            print(source)
            traceback.print_exc()
            raise

In [6]:
total_start = time.time()
for name in NOTEBOOKS:
    path = ROOT / name
    if not path.exists():
        print(f"[SKIP] {name} not found")
        continue
    print(f"=== Running {name} ===")
    start = time.time()
    run_notebook(path)
    print(f"=== Done {name} in {time.time() - start:.1f}s ===\n")

print(f"All notebooks finished in {time.time() - total_start:.1f}s")

=== Running 1_eda.ipynb ===
=== Done 1_eda.ipynb in 1.4s ===

=== Running 2_cross_tab_analysis.ipynb ===
=== Done 2_cross_tab_analysis.ipynb in 5.5s ===

=== Running 3_sig_analysis.ipynb ===
=== Done 3_sig_analysis.ipynb in 0.1s ===

=== Running 4_mlr_st.ipynb ===
=== Done 4_mlr_st.ipynb in 1.6s ===

=== Running 5_mlr_phq9.ipynb ===
=== Done 5_mlr_phq9.ipynb in 1.3s ===

=== Running 6_analysis_for_mind_well.ipynb ===
=== Done 6_analysis_for_mind_well.ipynb in 8.6s ===

All notebooks finished in 18.5s
